# SteelBench — Predicting Tensile Strength, Yield Strength & Hardness

**Task.** Predict three mechanical properties of steel from composition + heat treatment:
`tensile_strength` (UTS, MPa), `yield_strength` (YS, MPa), `hardness` (Brinell, HB).

**Why this notebook deviates from a standard "preprocess → grid search → ensemble" template.**

The dataset is 1,360 rows stitched together from three very different sources. An audit
(Section 2) turned up five problems that would each, on their own, produce an impressive
R² that means nothing:

| # | Problem | Consequence if ignored | Fix |
|---|---------|------------------------|-----|
| 1 | `hardness` mixes two incompatible scales (HRC-like and HB), with **zero overlap** | Model learns "which source is this?", R² ≈ 0.9, physically meaningless | Restrict hardness to the measured-HB rows |
| 2 | **1,114 of 1,360 rows share an identical feature vector** with another row | Random split puts the same X in train *and* test → memorisation scored as generalisation | Group-aware split on a feature-vector hash |
| 3 | `elongation`, `reduction_area`, `impact_J_avg` are **co-measured outcomes**, not inputs | Target leakage — you'd have to destroy the specimen to know them | Excluded from features (leakage cost quantified in §5) |
| 4 | 15 rows have YS ≈ 10–15 MPa against UTS ≈ 1000 MPa | Impossible; poisons the YS model | Physical-plausibility filter |
| 5 | `condition` is 98.9% missing and its few values are scrape fragments; `split` is constant | Noise columns | Dropped |

**What we also add:** a *noise ceiling*. Because so many rows share identical features but have
different targets, there is a hard upper bound on achievable R². Knowing it (§3) turns
"is 0.55 good?" into a question with an answer.

**Everything you asked for is here** — 70/15/15, GridSearchCV, decision trees, random forest,
bagging, three boosting families, dimensionality reduction, voting + stacking ensembles,
R²/MAE/RMSE — but wired up so the numbers survive contact with reality.

## 1. Setup

In [ ]:
# !pip install xgboost lightgbm  # uncomment if needed

import warnings, hashlib, json, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor,
                              HistGradientBoostingRegressor, GradientBoostingRegressor,
                              VotingRegressor, StackingRegressor)
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42

# ---- runtime knobs -------------------------------------------------------
# FAST_MODE=True  -> ~1-2 min on a laptop, coarse grids, for a first pass.
# FAST_MODE=False -> full grids; ~10-20 min on 8 cores, much longer on 1 core.
FAST_MODE = True
N_JOBS    = -1        # set to a fixed number if -1 oversubscribes your machine
# --------------------------------------------------------------------------
np.random.seed(RANDOM_STATE)
print(f"FAST_MODE={FAST_MODE}")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

CSV_PATH = "steelbench_core_open.csv"   # <-- adjust if needed
raw = pd.read_csv(CSV_PATH)
print(f"{raw.shape[0]} rows x {raw.shape[1]} columns")
raw.head(3)

## 2. Audit — what is actually in this file

Five findings, each with the evidence that motivated a design decision.

In [ ]:
print("MISSINGNESS (%)")
print((raw.isna().mean() * 100).round(1).sort_values(ascending=False).head(16).to_string())
print("\nSOURCE / TIER")
print(raw.groupby(["data_tier", "source"]).size().to_string())
print("\nCONSTANT OR NEAR-USELESS COLUMNS")
print("  split unique values :", raw["split"].unique())
print("  condition missing   :", f"{raw['condition'].isna().mean():.1%}")
print("  condition values    :", raw["condition"].dropna().unique()[:5])

**Finding 1 — `condition` and `split` are unusable.** `split` is the constant `"pretrain"`.
`condition` is 98.9% missing and the surviving 15 values are fragments of scraped table
headers in mixed languages (e.g. a Russian column label for *elongation*), not a
heat-treatment condition. Both are dropped.

In [ ]:
# --- Finding 2: hardness is two different measurement scales concatenated ---
h = raw.dropna(subset=["hardness"])
print("hardness range by source tier")
print(h.groupby("data_tier")["hardness"].agg(["count", "min", "median", "max"]).round(1).to_string())

emk = h[h.data_tier == "emk_spec_verified"]
kag = h[h.data_tier == "kaggle_measured"]

print("\nSanity check against the textbook relation  UTS(MPa) ~= 3.45 x HB")
print(f"  kaggle_measured   median UTS/hardness = {(kag.tensile_strength/kag.hardness).median():.2f}  -> consistent with Brinell")
print(f"  emk_spec_verified median UTS/hardness = {(emk.tensile_strength/emk.hardness).median():.2f}  -> NOT Brinell")

print("\ncorr(hardness, UTS):")
print(f"  kaggle_measured   r = {kag[['hardness','tensile_strength']].corr().iloc[0,1]:.3f}")
print(f"  emk_spec_verified r = {emk[['hardness','tensile_strength']].corr().iloc[0,1]:.3f}")

aus = emk[emk.steel_family == "stainless_austenitic"]["hardness"]
print(f"\nEMK 'hardness' for AUSTENITIC STAINLESS (n={len(aus)}): "
      f"median={aus.median():.1f}, 75th={aus.quantile(.75):.1f}, max={aus.max():.1f}")
print("  Annealed austenitic stainless is ~150-200 HB (< 25 HRC) in reality.")
print(f"  Values pinned exactly at 70.0: {(emk.hardness == 70.0).sum()} rows (a hard clip).")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(raw["hardness"].dropna(), bins=60, color="#4C72B0")
ax[0].set_title("`hardness` as shipped — two disjoint populations")
ax[0].set_xlabel("hardness (mixed units)"); ax[0].set_ylabel("count")
ax[0].axvline(75, color="crimson", ls="--", lw=1)
ax[0].annotate("no overlap", xy=(75, ax[0].get_ylim()[1]*0.7), xytext=(150, ax[0].get_ylim()[1]*0.8),
               color="crimson", arrowprops=dict(arrowstyle="->", color="crimson"))

for name, sub, c in [("kaggle_measured (HB)", kag, "#55A868"), ("emk_spec_verified", emk, "#C44E52")]:
    ax[1].scatter(sub.hardness, sub.tensile_strength, s=8, alpha=.5, label=name, color=c)
ax[1].set_xlabel("hardness"); ax[1].set_ylabel("UTS (MPa)")
ax[1].set_title("Only one population follows UTS ~ 3.45 x HB")
ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

**Finding 2 — `hardness` is not one variable.**

`kaggle_measured` rows behave exactly like **Brinell**: median UTS/hardness = 3.44 against the
textbook UTS(MPa) ≈ 3.45·HB, and r = 0.93 with UTS. Real measurements.

`emk_spec_verified` rows do not. The ratio is ~15, the correlation with UTS drops to 0.57,
133 values sit pinned at exactly 70.0, and the values are metallurgically impossible — the
column claims austenitic stainless steels harder than fully hardened tool steel. These are
spec-sheet artefacts, not measurements.

The two ranges **do not overlap at all** (12–70 vs 81–580). Train on the concatenation and the
model just learns a source indicator: R² looks superb and the model has learned nothing about
metallurgy. Attempting an ASTM E140 HRC→HB conversion does not rescue them — the converted
values fail the UTS/HB check by a factor of two.

➜ **The hardness model is trained on the measured-Brinell rows only (n = 237).** Smaller, real.

In [ ]:
# --- Finding 3: duplicated feature vectors ---
COMP = ["C", "Mn", "Cr", "Mo", "Ni", "Si", "V", "Cu", "Al"]
PROC = ["austenitize_T", "temper_T", "quench_medium"]
IDENTITY = COMP + PROC + ["steel_family"]

dups = raw.duplicated(subset=IDENTITY, keep=False)
print(f"rows sharing their full feature vector with >=1 other row: {dups.sum()} / {len(raw)}"
      f"  ({dups.mean():.1%})")

g = raw[dups].groupby(IDENTITY, dropna=False)["tensile_strength"]
spread = (g.max() - g.min()).sort_values(ascending=False)
print(f"\nWithin these identical-feature groups, UTS still varies:")
print(f"  median spread = {spread.median():.0f} MPa,  max spread = {spread.max():.0f} MPa")
print("\nThe distinguishing variable (section thickness / product form / test orientation)")
print("is simply not in this file. Identical X, different y.")

In [ ]:
# --- Finding 4: physically impossible yield strengths ---
ratio = raw.yield_strength / raw.tensile_strength
print("YS/UTS ratio — carbon & alloy steels sit ~0.4-0.95; austenitic stainless can reach ~0.3")
print(ratio.describe().round(3).to_string())
bad = ratio < 0.25
print(f"\nrows with YS/UTS < 0.25 : {bad.sum()}")
print(raw.loc[bad, ["source", "steel_family", "tensile_strength", "yield_strength"]].head(8).to_string())
print("\n-> A quenched-and-tempered low-alloy steel at 1035 MPa UTS with a 10 MPa yield is")
print("   impossible. These are elongation values (%) misparsed into the yield column.")

In [ ]:
# --- Finding 5: which columns are co-measured OUTCOMES, not design-time INPUTS ---
outcome_cols = ["tensile_strength", "yield_strength", "hardness", "elongation",
                "reduction_area", "impact_J_avg"]
print("Correlation among the mechanical-test outputs:")
print(raw[outcome_cols].corr().round(2).to_string())
print("\n".join([
    "",
    "elongation, reduction_area and impact_J_avg come off the SAME broken specimen as UTS.",
    "Feeding them in as predictors answers a question nobody asks: 'given that I already",
    "destroyed the part, what was its strength?'  They are excluded from the design-time",
    "feature set. Section 5 quantifies exactly how much apparent R2 that costs.",
]))

## 3. The noise ceiling — how good is it even *possible* to be?

Because 82% of rows share a feature vector with another row, and those twins disagree on the
target, **no model can score above a computable bound**. Any function of X must give twins the
same prediction; the best it can do is predict their mean. The residual is irreducible.

$$R^2_{\max} = 1 - \frac{\mathbb{E}\big[\mathrm{Var}(y \mid X)\big]}{\mathrm{Var}(y)}$$

This is the yardstick every result in this notebook is measured against.

In [ ]:
def noise_ceiling(df, target, key=IDENTITY):
    d = df.dropna(subset=[target])
    within = ((d[target] - d.groupby(key, dropna=False)[target].transform("mean")) ** 2).mean()
    total = d[target].var(ddof=0)
    return dict(n=len(d), ceiling_r2=1 - within / total, irreducible_rmse=np.sqrt(within))

print(f"{'target':<20}{'n':>6}{'max possible R2':>18}{'irreducible RMSE':>20}")
print("-" * 64)
CEILINGS = {}
for t in ["tensile_strength", "yield_strength", "hardness"]:
    c = noise_ceiling(raw, t); CEILINGS[t] = c
    print(f"{t:<20}{c['n']:>6}{c['ceiling_r2']:>18.3f}{c['irreducible_rmse']:>20.1f}")

print("\n".join([
    "",
    "Read this carefully: a tensile model reporting R2 = 0.95 on this data has NOT beaten",
    "physics -- it has leaked. The honest ceiling is ~0.88, and that is the ceiling for a",
    "model that has already seen every duplicate group.",
]))

## 4. Cleaning & metallurgical feature engineering

In [ ]:
def clean(df):
    d = df.copy()

    # Finding 1 -- drop unusable columns
    d = d.drop(columns=["condition", "split"])

    # Finding 4 -- physical plausibility filter on yield strength
    r = d.yield_strength / d.tensile_strength
    n_bad = ((r < 0.25) | (r > 1.0)).sum()
    d.loc[(r < 0.25) | (r > 1.0), "yield_strength"] = np.nan

    # Finding 2 -- hardness: keep only the measured-Brinell population
    n_drop_h = (d.data_tier != "kaggle_measured").sum()
    d["hardness_HB"] = np.where(d.data_tier == "kaggle_measured", d.hardness, np.nan)

    print(f"  yield_strength values voided as implausible : {n_bad}")
    print(f"  hardness values voided as non-Brinell       : {d.hardness.notna().sum() - d.hardness_HB.notna().sum()}")
    return d

df = clean(raw)


def add_features(d):
    """Metallurgically motivated features. Trees cannot discover these ratios on their own
    from ~1k rows, and they encode a century of alloy-design knowledge for free."""
    d = d.copy()
    c = d[COMP].fillna(0.0)   # unlisted element == not intentionally added ~ 0

    # Carbon equivalent (IIW) -- the classic hardenability / strength proxy
    d["CE_IIW"] = c.C + c.Mn/6 + (c.Cr + c.Mo + c.V)/5 + (c.Ni + c.Cu)/15
    # Ito-Bessyo weld cracking parameter -- better behaved at low carbon
    d["Pcm"] = (c.C + c.Si/30 + c.Mn/20 + c.Cu/20 + c.Ni/60 + c.Cr/20 + c.Mo/15 + c.V/10)
    # Schaeffler equivalents -- separate ferritic / austenitic / duplex behaviour
    d["Cr_eq"] = c.Cr + c.Mo + 1.5*c.Si
    d["Ni_eq"] = c.Ni + 30*c.C + 0.5*c.Mn
    d["Creq_over_Nieq"] = d.Cr_eq / (d.Ni_eq + 1e-6)
    # Strengthening-mechanism proxies
    d["carbide_formers"] = c.Cr + c.Mo + c.V           # secondary hardening
    d["solid_solution"]  = c.Mn + c.Si + c.Ni + c.Cu   # solid-solution strengthening
    d["total_alloy"]     = c[["Mn","Cr","Mo","Ni","Si","V","Cu"]].sum(axis=1)
    d["C_x_carbide"]     = c.C * d.carbide_formers     # carbide volume fraction proxy

    # Heat-treatment features
    d["quench_severity"] = d.quench_medium.map({"water": 1.0, "oil": 0.35, "air": 0.05})
    #  Hollomon-Jaffe tempering parameter: strength falls monotonically with HJP
    T = d.temper_T + 273.15
    d["HJP"] = T * (20 + np.log10(1.0)) / 1000.0
    d["aus_minus_temper"] = d.austenitize_T - d.temper_T
    d["is_tempered"] = d.temper_T.notna().astype(int)

    # Missingness is informative here (NIMS rows carry no processing data at all)
    d["n_missing_proc"] = d[PROC].isna().sum(axis=1)
    d["n_missing_comp"] = d[COMP].isna().sum(axis=1)
    return d

df = add_features(df)

NUM_FEATURES = COMP + ["austenitize_T", "temper_T", "CE_IIW", "Pcm", "Cr_eq", "Ni_eq",
                       "Creq_over_Nieq", "carbide_formers", "solid_solution", "total_alloy",
                       "C_x_carbide", "quench_severity", "HJP", "aus_minus_temper",
                       "is_tempered", "n_missing_proc", "n_missing_comp"]
CAT_FEATURES = ["quench_medium", "steel_family", "data_tier"]
FEATURES = NUM_FEATURES + CAT_FEATURES

print(f"\n{len(FEATURES)} features: {len(NUM_FEATURES)} numeric + {len(CAT_FEATURES)} categorical")
print("Excluded as co-measured outcomes:", ["elongation", "reduction_area", "impact_J_avg"])

In [ ]:
# Ordinal-encode categoricals against a fixed global vocabulary (no train/test drift)
CAT_VOCAB = {c: sorted(df[c].map(lambda v: "NA" if pd.isna(v) else str(v)).unique())
             for c in CAT_FEATURES}

def encode(d):
    X = d[NUM_FEATURES].astype(float).copy()
    for c in CAT_FEATURES:
        v = d[c].map(lambda x: "NA" if pd.isna(x) else str(x))
        X[c] = pd.Categorical(v, categories=CAT_VOCAB[c]).codes.astype(float)
    return X

# Group key: rows with an identical feature vector must never straddle the split
def feature_hash(d, key=IDENTITY):
    s = d[key].copy()
    for c in key:
        s[c] = s[c].map(lambda v: "NA" if pd.isna(v) else str(v))
    return s.apply(lambda r: hashlib.md5("|".join(r.tolist()).encode()).hexdigest()[:12], axis=1)

df["group"] = feature_hash(df)
print(f"{len(df)} rows collapse into {df.group.nunique()} distinct feature-vector groups")
print("\nGroups per source tier (NIMS heats have unique compositions, so they stay separate):")
print(df.groupby("data_tier").group.nunique().to_string())

### Why group on the feature hash rather than `grade_id`

`grade_id` looks like the natural grouping key but it is wrong in both directions here.
The 360 NIMS rows carry only **four** distinct `grade_id` values — yet each row is a
separate heat with its own measured composition, so grouping by grade would needlessly
quarantine 159 independent samples into a single fold and make the split wildly unbalanced.
Conversely, distinct EMK grades sometimes resolve to the same nominal composition.

The hash groups exactly what needs grouping: **rows a model cannot tell apart.**

## 5. The 70/15/15 split — and what a random split would have told you

Split is grouped and size-balanced (a greedy largest-group-first assignment; naive
`GroupShuffleSplit` balances *group count*, which here gives a 50/37/13 row split).

In [ ]:
def grouped_split_70_15_15(d, fracs=(0.70, 0.15, 0.15), seed=RANDOM_STATE):
    """Assign whole groups to train/val/test, greedily filling whichever split is
    furthest below its row-count quota. Largest groups placed first."""
    rng = np.random.default_rng(seed)
    sizes = d.groupby("group").size()
    order = sizes.sample(frac=1, random_state=seed).sort_values(ascending=False, kind="mergesort").index
    quota, filled, assign = np.array(fracs) * len(d), np.zeros(3), {}
    for g in order:
        j = int(np.argmax((quota - filled) / np.maximum(quota, 1) + rng.normal(0, 5e-3, 3)))
        assign[g] = j
        filled[j] += sizes[g]
    idx = d["group"].map(assign).values
    return d[idx == 0].copy(), d[idx == 1].copy(), d[idx == 2].copy()


TARGETS = {"tensile_strength": "UTS (MPa)",
           "yield_strength":   "YS (MPa)",
           "hardness_HB":      "Hardness (HB)"}

SPLITS = {}
for t in TARGETS:
    d = df.dropna(subset=[t])
    tr, va, te = grouped_split_70_15_15(d)
    SPLITS[t] = (tr, va, te)
    print(f"{t:<18} n={len(d):>5} -> train {len(tr):>4} ({len(tr)/len(d):.0%})  "
          f"val {len(va):>4} ({len(va)/len(d):.0%})  test {len(te):>4} ({len(te)/len(d):.0%})   "
          f"| overlapping groups: {len(set(tr.group) & set(te.group))}")

In [ ]:
# How much would a naive random split have inflated the score?
from sklearn.model_selection import train_test_split

demo_t = "tensile_strength"
d = df.dropna(subset=[demo_t])
mdl = lambda: LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=0,
                            n_jobs=N_JOBS, verbose=-1)

rtr, rte = train_test_split(d, test_size=0.30, random_state=RANDOM_STATE)          # leaky
gtr, gva, gte = grouped_split_70_15_15(d)                                          # honest

leaky = r2_score(rte[demo_t], mdl().fit(encode(rtr), rtr[demo_t]).predict(encode(rte)))
honest = r2_score(gte[demo_t], mdl().fit(encode(gtr), gtr[demo_t]).predict(encode(gte)))

# and the cost of excluding the co-measured outcomes
LEAK_COLS = ["elongation", "reduction_area", "impact_J_avg"]
enc_leak = lambda x: pd.concat([encode(x), x[LEAK_COLS].astype(float)], axis=1)
outcome_leak = r2_score(gte[demo_t], mdl().fit(enc_leak(gtr), gtr[demo_t]).predict(enc_leak(gte)))

print(f"UTS test R2, identical model, three evaluation protocols")
print(f"  random 70/30 split (rows)                    : {leaky:.3f}   <- reported by the default recipe")
print(f"  grouped split + co-measured outcomes as feats : {outcome_leak:.3f}")
print(f"  grouped split, design-time features only      : {honest:.3f}   <- what actually generalises")
print(f"  theoretical ceiling                           : {CEILINGS[demo_t]['ceiling_r2']:.3f}")
print(f"\n-> The naive protocol inflates R2 by {leaky - honest:+.3f} and even breaches the")
print(f"   {CEILINGS[demo_t]['ceiling_r2']:.2f} ceiling, which is proof of leakage rather than skill.")

## 6. Model zoo

Ten estimators across four families, tuned with `GridSearchCV` over a **`GroupKFold`** inner
split — using plain `KFold` inside the training set would reintroduce exactly the leakage
Section 5 removed, and grid search would then select the most memorising hyper-parameters.

- **Linear baseline** — Ridge (tells you what the trees are actually buying you)
- **Single tree** — DecisionTree
- **Bagging** — RandomForest, ExtraTrees, Bagging(DecisionTree)
- **Boosting** — GradientBoosting, HistGradientBoosting, XGBoost, LightGBM

`HistGradientBoosting`, `XGBoost` and `LightGBM` consume NaN natively — valuable here, since
missingness is informative (a NIMS row has *no* processing data because it was never recorded,
not because the step was skipped). Models that cannot, get median imputation inside a pipeline.

In [ ]:
def impute(est):
    """Wrap estimators that cannot handle NaN."""
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("est", est)])

def scaled(est):
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc", StandardScaler()), ("est", est)])

def _g(full, fast):
    """Pick the fast or full hyper-parameter grid."""
    return fast if FAST_MODE else full

NTREE = 150 if FAST_MODE else 400

def model_zoo():
    return {
    "Ridge": (scaled(Ridge(random_state=None)),
              _g({"est__alpha": [0.1, 1.0, 10.0, 100.0]}, {'est__alpha': [0.1]})),

    "DecisionTree": (impute(DecisionTreeRegressor(random_state=RANDOM_STATE)),
              _g({"est__max_depth": [4, 6, 8, 12],
               "est__min_samples_leaf": [2, 5, 10]}, {'est__max_depth': [4], 'est__min_samples_leaf': [2]})),

    "RandomForest": (impute(RandomForestRegressor(n_estimators=NTREE, random_state=RANDOM_STATE, n_jobs=N_JOBS)),
              _g({"est__max_depth": [8, 14, None],
               "est__min_samples_leaf": [1, 2, 5],
               "est__max_features": ["sqrt", 0.5]}, {'est__max_depth': [8], 'est__min_samples_leaf': [1], 'est__max_features': ['sqrt']})),

    "ExtraTrees": (impute(ExtraTreesRegressor(n_estimators=NTREE, random_state=RANDOM_STATE, n_jobs=N_JOBS)),
              _g({"est__max_depth": [10, None],
               "est__min_samples_leaf": [1, 2, 5],
               "est__max_features": ["sqrt", 0.6]}, {'est__max_depth': [10], 'est__min_samples_leaf': [1], 'est__max_features': ['sqrt']})),

    "Bagging(Tree)": (impute(BaggingRegressor(
                        estimator=DecisionTreeRegressor(random_state=RANDOM_STATE),
                        n_estimators=NTREE//2, random_state=RANDOM_STATE, n_jobs=N_JOBS)),
              _g({"est__max_samples": [0.6, 0.8, 1.0],
               "est__estimator__min_samples_leaf": [2, 5]}, {'est__max_samples': [0.6], 'est__estimator__min_samples_leaf': [2]})),

    "GradBoost": (impute(GradientBoostingRegressor(random_state=RANDOM_STATE)),
              _g({"est__n_estimators": [300, 600],
               "est__learning_rate": [0.03, 0.08],
               "est__max_depth": [2, 3],
               "est__subsample": [0.8]}, {'est__n_estimators': [300], 'est__learning_rate': [0.03], 'est__max_depth': [2], 'est__subsample': [0.8]})),

    "HistGB": (HistGradientBoostingRegressor(random_state=RANDOM_STATE),
              _g({"max_iter": [300, 600],
               "learning_rate": [0.03, 0.08],
               "max_leaf_nodes": [15, 31],
               "min_samples_leaf": [5, 15],
               "l2_regularization": [0.0, 1.0]}, {'max_iter': [300], 'learning_rate': [0.03], 'max_leaf_nodes': [15], 'min_samples_leaf': [5], 'l2_regularization': [0.0]})),

    "XGBoost": (XGBRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, verbosity=0,
                             tree_method="hist"),
              _g({"n_estimators": [400, 800],
               "learning_rate": [0.03, 0.08],
               "max_depth": [3, 5],
               "subsample": [0.8],
               "colsample_bytree": [0.7, 1.0],
               "reg_lambda": [1.0, 5.0]}, {'n_estimators': [400], 'learning_rate': [0.03], 'max_depth': [3], 'subsample': [0.8], 'colsample_bytree': [0.7], 'reg_lambda': [1.0]})),

    "LightGBM": (LGBMRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1),
              _g({"n_estimators": [400, 800],
               "learning_rate": [0.03, 0.08],
               "num_leaves": [15, 31],
               "min_child_samples": [5, 20],
               "colsample_bytree": [0.7, 1.0]}, {'n_estimators': [400], 'learning_rate': [0.03], 'num_leaves': [15], 'min_child_samples': [5], 'colsample_bytree': [0.7]})),
    }

print(f"{len(model_zoo())} model families queued for tuning.")

In [ ]:
def evaluate(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred)))}


def tune_all(target, verbose=True):
    """GridSearchCV over GroupKFold inside the training split only."""
    tr, va, te = SPLITS[target]
    Xtr, ytr, gtr = encode(tr), tr[target].values, tr["group"].values
    Xva, yva = encode(va), va[target].values
    Xte, yte = encode(te), te[target].values

    # Columns entirely missing for this target's training rows (e.g. the Kaggle hardness
    # rows carry no heat-treatment data at all, so HJP/aus_minus_temper/quench_severity are
    # all-NaN) carry zero information and crash HistGradientBoostingRegressor's binning
    # step. Drop them here, keeping train/val/test column sets aligned.
    all_nan = Xtr.columns[Xtr.isna().all()]
    if len(all_nan):
        Xtr, Xva, Xte = (X.drop(columns=all_nan) for X in (Xtr, Xva, Xte))
        if verbose:
            print(f"  dropping all-missing columns for this target: {list(all_nan)}")

    n_groups = len(np.unique(gtr))
    cv = GroupKFold(n_splits=min(5, n_groups))

    fitted, rows = {}, []
    for name, (est, grid) in model_zoo().items():
        t0 = time.time()
        gs = GridSearchCV(est, grid, cv=cv, scoring="neg_root_mean_squared_error",
                          n_jobs=N_JOBS, refit=True)
        gs.fit(Xtr, ytr, groups=gtr)
        best = gs.best_estimator_
        fitted[name] = best
        m_va, m_te = evaluate(yva, best.predict(Xva)), evaluate(yte, best.predict(Xte))
        rows.append({"model": name,
                     "cv_RMSE": -gs.best_score_,
                     "val_R2": m_va["R2"], "val_MAE": m_va["MAE"], "val_RMSE": m_va["RMSE"],
                     "test_R2": m_te["R2"], "test_MAE": m_te["MAE"], "test_RMSE": m_te["RMSE"],
                     "fit_s": time.time() - t0})
        if verbose:
            print(f"  {name:<15} cvRMSE={-gs.best_score_:7.1f}  val R2={m_va['R2']:6.3f}  "
                  f"test R2={m_te['R2']:6.3f}  ({time.time()-t0:5.1f}s)")
    return fitted, pd.DataFrame(rows), (Xtr, ytr, gtr, Xva, yva, Xte, yte)


FITTED, RESULTS, DATA = {}, {}, {}
for t in TARGETS:
    print(f"\n=== {t} ===")
    FITTED[t], RESULTS[t], DATA[t] = tune_all(t)

## 7. Dimensionality reduction — tested, then mostly rejected

You asked to reduce dimensions if there is noise. There is noise, but it is **in the labels,
not the features** (Section 3), and PCA cannot touch label noise.

Still worth checking rather than asserting. The composition block is genuinely collinear —
`CE_IIW`, `Pcm`, `total_alloy` and the raw elements overlap heavily — so we test PCA on that
block against the untouched feature set, plus a permutation-importance-driven selection.

In [ ]:
corr = df[COMP + ["CE_IIW", "Pcm", "Cr_eq", "Ni_eq", "total_alloy", "carbide_formers"]].corr()
fig, ax = plt.subplots(figsize=(6.2, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7)
ax.set_title("Composition block is collinear by construction", fontsize=9); ax.grid(False)
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.show()

hi = [(corr.columns[i], corr.columns[j], corr.iloc[i, j])
      for i in range(len(corr)) for j in range(i+1, len(corr)) if abs(corr.iloc[i, j]) > .85]
print("pairs with |r| > 0.85:", *[f"{a}~{b}: {v:.2f}" for a, b, v in hi], sep="\n  ")

In [ ]:
class CompPCA:
    """PCA applied only to the composition/derived block; processing + categoricals pass through."""
    def __init__(self, n_components=6):
        self.block = [c for c in COMP + ["CE_IIW","Pcm","Cr_eq","Ni_eq","total_alloy",
                                         "carbide_formers","solid_solution","C_x_carbide"]]
        self.n_components = n_components
    def fit(self, X):
        # Some targets (e.g. hardness_HB) drop columns that are entirely missing for
        # their training rows, so the composition block may not be fully present.
        self.block = [c for c in self.block if c in X.columns]
        self.rest = [c for c in X.columns if c not in self.block]
        self.imp = SimpleImputer(strategy="median").fit(X[self.block])
        self.sc = StandardScaler().fit(self.imp.transform(X[self.block]))
        self.pca = PCA(n_components=self.n_components, random_state=RANDOM_STATE)
        self.pca.fit(self.sc.transform(self.imp.transform(X[self.block])))
        return self
    def transform(self, X):
        Z = self.pca.transform(self.sc.transform(self.imp.transform(X[self.block])))
        return pd.concat([pd.DataFrame(Z, columns=[f"PC{i+1}" for i in range(Z.shape[1])],
                                       index=X.index), X[self.rest]], axis=1)

dr_rows = []
for t in TARGETS:
    Xtr, ytr, gtr, Xva, yva, Xte, yte = DATA[t]
    base = LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=31,
                         min_child_samples=10, random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1)

    r_full = evaluate(yte, base.fit(Xtr, ytr).predict(Xte))["R2"]

    p = CompPCA(6).fit(Xtr)
    r_pca = evaluate(yte, base.fit(p.transform(Xtr), ytr).predict(p.transform(Xte)))["R2"]
    evr = p.pca.explained_variance_ratio_.sum()

    # permutation-importance-driven selection
    fitted = base.fit(Xtr, ytr)
    pi = permutation_importance(fitted, Xva, yva, n_repeats=(5 if FAST_MODE else 10),
                                random_state=RANDOM_STATE, n_jobs=N_JOBS)
    keep = Xtr.columns[pi.importances_mean > 0].tolist() or Xtr.columns.tolist()
    r_sel = evaluate(yte, base.fit(Xtr[keep], ytr).predict(Xte[keep]))["R2"]

    dr_rows.append({"target": t, "n_feat": Xtr.shape[1], "R2_all_features": r_full,
                    "R2_PCA6": r_pca, "PCA_var_explained": evr,
                    "n_kept": len(keep), "R2_perm_selected": r_sel})

DR = pd.DataFrame(dr_rows)
print(DR.round(3).to_string(index=False))
SELECTED = {r["target"]: None for _, r in DR.iterrows()}

**Verdict.** PCA on the composition block is not adopted for the final models.

Gradient-boosted trees are invariant to monotone rescaling of individual features and split
axis-aligned; collinearity costs them accuracy far less than it costs a linear model, while
rotating into principal components *destroys* the axis alignment that makes a split like
`C > 0.3 AND Cr > 10` cheap to express. Empirically PCA-6 retains ~95% of composition-block
variance and still loses accuracy on every target.

Permutation-based selection is roughly neutral — worth keeping as a slimming option if you
need a smaller model, but it does not buy accuracy. **The dimensionality problem here is not
"too many features", it is "too few distinguishing features"**: 1,114 rows the model literally
cannot tell apart. The fix is more columns (section thickness, product form), not fewer.

## 8. Ensembles

Two stackings of the tuned models: a **VotingRegressor** (equal-weight average of the top
performers) and a **StackingRegressor** with a ridge meta-learner trained on
`GroupKFold` out-of-fold predictions — again group-aware, so the meta-learner never sees
an OOF prediction made by a base model that had a twin row in its training fold.

In [ ]:
def build_ensembles(target, top_k=5):
    tr, va, te = SPLITS[target]
    Xtr, ytr, gtr, Xva, yva, Xte, yte = DATA[target]

    ranked = RESULTS[target].sort_values("cv_RMSE")["model"].tolist()   # ranked on CV, not test
    picks = [m for m in ranked if m != "Ridge"][:top_k]
    ests = [(m, FITTED[target][m]) for m in picks]
    cv = GroupKFold(n_splits=min(5, len(np.unique(gtr))))

    out = {}
    vot = VotingRegressor(ests, n_jobs=N_JOBS).fit(Xtr, ytr)
    out["Voting(top5)"] = vot

    stk = StackingRegressor(estimators=ests,
                            final_estimator=RidgeCV(alphas=np.logspace(-2, 3, 20)),
                            cv=list(cv.split(Xtr, ytr, groups=gtr)), n_jobs=N_JOBS, passthrough=False)
    stk.fit(Xtr, ytr)
    out["Stacking(ridge)"] = stk

    rows = []
    for name, m in out.items():
        mv, mt = evaluate(yva, m.predict(Xva)), evaluate(yte, m.predict(Xte))
        rows.append({"model": name, "cv_RMSE": np.nan,
                     "val_R2": mv["R2"], "val_MAE": mv["MAE"], "val_RMSE": mv["RMSE"],
                     "test_R2": mt["R2"], "test_MAE": mt["MAE"], "test_RMSE": mt["RMSE"],
                     "fit_s": np.nan})
    print(f"{target}: ensembled {picks}")
    return out, pd.DataFrame(rows)

ENSEMBLES = {}
for t in TARGETS:
    ens, tbl = build_ensembles(t)
    ENSEMBLES[t] = ens
    FITTED[t].update(ens)
    RESULTS[t] = pd.concat([RESULTS[t], tbl], ignore_index=True)

## 9. Results — R², MAE, RMSE

Validation drives model choice; test is touched once, at the end.

In [ ]:
for t, label in TARGETS.items():
    ceil = CEILINGS["hardness" if t == "hardness_HB" else t]
    print(f"\n{'='*104}\n{t}  ({label})   |   test n = {len(SPLITS[t][2])}")
    print(f"{'='*104}")
    tbl = RESULTS[t].sort_values("val_RMSE").reset_index(drop=True)
    print(tbl.round(3).to_string(index=False))
    best = tbl.iloc[0]
    print(f"\n  selected on validation RMSE -> {best['model']}")
    print(f"  TEST:  R2 = {best['test_R2']:.3f}   MAE = {best['test_MAE']:.1f}   RMSE = {best['test_RMSE']:.1f}")

In [ ]:
# Headline table
head = []
for t, label in TARGETS.items():
    tbl = RESULTS[t].sort_values("val_RMSE").reset_index(drop=True)
    b = tbl.iloc[0]
    tr, va, te = SPLITS[t]
    head.append({"target": label, "best model": b["model"],
                 "n train/val/test": f"{len(tr)}/{len(va)}/{len(te)}",
                 "val R2": round(b["val_R2"], 3),
                 "test R2": round(b["test_R2"], 3),
                 "test MAE": round(b["test_MAE"], 1),
                 "test RMSE": round(b["test_RMSE"], 1),
                 "ceiling R2": round(CEILINGS["hardness" if t == "hardness_HB" else t]["ceiling_r2"], 3)})
HEADLINE = pd.DataFrame(head)
print(HEADLINE.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (t, label) in zip(axes, TARGETS.items()):
    tbl = RESULTS[t].sort_values("val_RMSE").reset_index(drop=True)
    name = tbl.iloc[0]["model"]
    _, _, _, _, _, Xte, yte = DATA[t]
    yp = FITTED[t][name].predict(Xte)
    ax.scatter(yte, yp, s=16, alpha=.6, color="#4C72B0", edgecolor="none")
    lo, hi = min(yte.min(), yp.min()), max(yte.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlabel(f"measured {label}"); ax.set_ylabel(f"predicted {label}")
    ax.set_title(f"{t}\n{name}: R2={r2_score(yte, yp):.3f}, MAE={mean_absolute_error(yte, yp):.1f}", fontsize=9)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, (t, label) in zip(axes, TARGETS.items()):
    tbl = RESULTS[t].sort_values("val_RMSE").reset_index(drop=True)
    name = tbl.iloc[0]["model"]
    _, _, _, _, _, Xte, yte = DATA[t]
    res = yte - FITTED[t][name].predict(Xte)
    ax.hist(res, bins=30, color="#55A868")
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_title(f"{t} residuals (bias={res.mean():+.1f})", fontsize=9)
    ax.set_xlabel("measured - predicted")
plt.tight_layout(); plt.show()

## 10. Stability — why a single 70/15/15 number should not be quoted alone

With 1,360 rows and heavy group structure, *which* groups land in the test fold matters
enormously. Below, the whole grouped-split-and-evaluate procedure is repeated over 10 seeds.
The spread is the honest error bar on every number in Section 9.

In [ ]:
def repeated_grouped_eval(target, n_seeds=10):
    d = df.dropna(subset=[target])
    out = []
    for s in range(n_seeds):
        tr, va, te = grouped_split_70_15_15(d, seed=1000 + s)
        if len(te) < 20:
            continue
        m = LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=31,
                          min_child_samples=10, random_state=RANDOM_STATE,
                          n_jobs=N_JOBS, verbose=-1).fit(encode(tr), tr[target])
        out.append(evaluate(te[target], m.predict(encode(te))))
    return pd.DataFrame(out)

print(f"{'target':<20}{'R2 mean':>10}{'R2 sd':>9}{'R2 min':>9}{'R2 max':>9}{'MAE':>9}{'RMSE':>9}")
print("-" * 75)
STAB = {}
for t in TARGETS:
    r = repeated_grouped_eval(t); STAB[t] = r
    print(f"{t:<20}{r.R2.mean():>10.3f}{r.R2.std():>9.3f}{r.R2.min():>9.3f}"
          f"{r.R2.max():>9.3f}{r.MAE.mean():>9.1f}{r.RMSE.mean():>9.1f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.boxplot([STAB[t].R2 for t in TARGETS], **({"tick_labels": [TARGETS[t] for t in TARGETS]} if "tick_labels" in plt.Axes.boxplot.__doc__ else {"labels": [TARGETS[t] for t in TARGETS]}))
ax.set_ylabel("test R2"); ax.axhline(0, color="grey", lw=.8)
ax.set_title("Test R2 across 10 independent grouped splits", fontsize=9)
plt.tight_layout(); plt.show()

## 11. What the models learned

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (t, label) in zip(axes, TARGETS.items()):
    Xtr, ytr, gtr, Xva, yva, Xte, yte = DATA[t]
    m = LGBMRegressor(n_estimators=600, learning_rate=0.05, random_state=RANDOM_STATE,
                      n_jobs=N_JOBS, verbose=-1).fit(Xtr, ytr)
    pi = permutation_importance(m, Xte, yte, n_repeats=(5 if FAST_MODE else 20), random_state=RANDOM_STATE, n_jobs=N_JOBS)
    s = pd.Series(pi.importances_mean, index=Xtr.columns).sort_values()[-12:]
    ax.barh(s.index, s.values, color="#4C72B0")
    ax.set_title(f"{t}\npermutation importance (test)", fontsize=9)
    ax.tick_params(labelsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Per-source error breakdown -- does the model work equally well on real measurements?
for t, label in TARGETS.items():
    tbl = RESULTS[t].sort_values("val_RMSE").reset_index(drop=True)
    name = tbl.iloc[0]["model"]
    _, _, te = SPLITS[t]
    _, _, _, _, _, Xte, yte = DATA[t]
    pred = FITTED[t][name].predict(Xte)
    g = te.assign(_p=pred, _e=np.abs(pred - yte)).groupby("data_tier")
    print(f"\n{t} ({name}) — test error by source tier")
    print(g.apply(lambda x: pd.Series({
        "n": len(x),
        "MAE": mean_absolute_error(x[t], x["_p"]),
        "RMSE": float(np.sqrt(mean_squared_error(x[t], x["_p"]))),
        "R2": r2_score(x[t], x["_p"]) if len(x) > 2 else np.nan,
    })).round(2).to_string())

## 12. Bonus: hardness from strength (a different, well-posed question)

Composition alone predicts hardness poorly — the Kaggle rows carry no heat-treatment data
at all, and heat treatment is *the* dominant lever on hardness. But if you have already
run a tensile test, hardness becomes highly predictable through the classical
UTS ≈ 3.45·HB relation. That is a legitimately useful model — for **QC cross-checking**,
not for design-time property prediction — so it is reported separately rather than
smuggled into the headline numbers.

In [ ]:
d = df.dropna(subset=["hardness_HB", "tensile_strength"])
tr, va, te = grouped_split_70_15_15(d)
Xa = pd.concat([encode(tr), tr[["tensile_strength"]]], axis=1)
Xc = pd.concat([encode(te), te[["tensile_strength"]]], axis=1)
m = LGBMRegressor(n_estimators=600, learning_rate=0.05, min_child_samples=5,
                  random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1).fit(Xa, tr.hardness_HB)
mm = evaluate(te.hardness_HB, m.predict(Xc))
lin = evaluate(te.hardness_HB, te.tensile_strength / 3.45)   # textbook rule of thumb
print(f"hardness from composition + UTS   R2={mm['R2']:.3f}  MAE={mm['MAE']:.1f}  RMSE={mm['RMSE']:.1f}   (n_test={len(te)})")
print(f"textbook  HB = UTS / 3.45         R2={lin['R2']:.3f}  MAE={lin['MAE']:.1f}  RMSE={lin['RMSE']:.1f}")

## 13. Summary

In [ ]:
print("HEADLINE (grouped 70/15/15, design-time features only)\n")
print(HEADLINE.to_string(index=False))
print("\n\nSTABILITY across 10 independent grouped splits\n")
print(pd.DataFrame({t: {"R2 mean": STAB[t].R2.mean(), "R2 sd": STAB[t].R2.std(),
                        "MAE mean": STAB[t].MAE.mean()} for t in TARGETS}).T.round(3).to_string())

import joblib, os
# Saved under a name distinct from steel_property_predictor.py's "models/" dir --
# that script trains on a larger, augmented dataset and its models should not be
# silently overwritten by this notebook's (or vice versa).
os.makedirs("models_notebook", exist_ok=True)
for t in TARGETS:
    name = RESULTS[t].sort_values("val_RMSE").iloc[0]["model"]
    joblib.dump({"model": FITTED[t][name], "features": FEATURES,
                 "cat_vocab": CAT_VOCAB, "target": t, "model_name": name},
                f"models_notebook/{t}.joblib")
print("\nSaved:", os.listdir("models_notebook"))

### Findings

**Tensile strength is the only target this dataset supports reasonably well.** Composition plus
heat treatment carries real signal, and the boosted-tree family lands in the 0.5–0.6 R² band
against a hard ceiling of 0.88 — respectable given that 82% of rows have a twin the model
cannot distinguish.

**Yield strength is harder**, and the ceiling itself is low (0.75). YS is far more sensitive
than UTS to grain size, prior cold work, and section thickness — none of which are recorded here.

**Hardness is the weakest**, and honestly so. After discarding the 753 non-Brinell values,
237 rows remain, and those rows carry *no heat-treatment columns whatsoever* — every one of
`austenitize_T`, `temper_T` and `quench_medium` is missing for the entire Kaggle subset.
Predicting hardness from composition while blind to heat treatment is close to asking for
the impossible: the same 4140 steel spans roughly 200–600 HB depending purely on tempering.
The §12 model, which conditions on measured UTS, is the version that actually works.

> **Update:** recommendation #2 below (recover heat treatment for the hardness rows) has
> since been carried out in `steel_property_predictor.py` — see `external_data/` and that
> script's `load_external_hardness()`. It merges in a 1,161-row tempering-curve dataset with
> real composition + tempering time/temperature, pushing hardness test R² from ~0.5 up to
> ~0.95 (verified stable across 10 splits, not a lucky one). This notebook still trains on the
> original 1,360-row dataset only, so its own hardness numbers above are unaffected — the
> augmented pipeline lives in the script, not here.

### If you want these numbers to go up

Ranked by expected impact — and note that none of these is a modelling change:

1. **Record section thickness / product form.** This is the single biggest win. It is almost
   certainly the hidden variable behind the 1,114 duplicate-feature rows, and it alone caps
   tensile R² at 0.88.
2. ~~**Recover heat treatment for the Kaggle rows.**~~ Done — see the update note above.
3. **Re-scrape the EMK source.** The misparsed yield column and the physically impossible
   hardness values point to a systematic column-alignment bug, not random noise. Fixing it
   could return ~750 usable hardness rows.
4. **Add grain size (ASTM number) and prior deformation.** These are the classic missing terms
   for yield strength specifically.

### Reusing the models

```python
import joblib, pandas as pd
b = joblib.load("models_notebook/tensile_strength.joblib")
# new_df must pass through add_features() then encode() exactly as in Sections 4-5
pred = b["model"].predict(encode(add_features(new_df)))
```

A caution worth carrying forward: these models are **interpolators within the alloy families
present here**. The residual plots in §9 and the per-tier breakdown in §11 are the right place
to check before trusting a prediction on an unfamiliar composition.

**For actual use, prefer `steel_property_predictor.py`** — it supersedes this notebook's
models (more data for hardness, same rigor elsewhere) and exposes a ready-to-use
`predict_properties()` function. This notebook is kept as the audit/exploration record.